<a href="https://colab.research.google.com/github/harshitadalne/AI-Practicals-Python-/blob/main/NNDL_Practical_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openpyxl tensorflow scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

!wget -q "$url" -O online_retail.zip
!unzip -o -q online_retail.zip

df = pd.read_excel("Online Retail.xlsx")

print("Dataset loaded successfully!")
print("Original dataset shape:", df.shape)
print("\nFirst 5 records:")
print(df.head())

df = df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
]

df = df.dropna(subset=["CustomerID"])

df = df[
    ~df["InvoiceNo"].astype(str).str.startswith("C")
]

df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df = df.dropna(subset=["Description"])

print("\nAfter cleaning:")
print("Dataset shape:", df.shape)

df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

print(
    "\nTotal Sales:",
    round(df["TotalAmount"].sum(), 2)
)

interaction_df = (
    df.groupby(["CustomerID", "StockCode"])
    .agg(
        Quantity=("Quantity", "sum"),
        TotalAmount=("TotalAmount", "sum")
    )
    .reset_index()
)

print("\nCustomer-Product interactions:")
print(interaction_df.head())

customers = interaction_df["CustomerID"].unique()
products = interaction_df["StockCode"].unique()

customer_to_index = {
    customer: index
    for index, customer in enumerate(customers)
}

product_to_index = {
    product: index
    for index, product in enumerate(products)
}

index_to_customer = {
    index: customer
    for customer, index in customer_to_index.items()
}

index_to_product = {
    index: product
    for product, index in product_to_index.items()
}

interaction_df["CustomerIndex"] = (
    interaction_df["CustomerID"]
    .map(customer_to_index)
)

interaction_df["ProductIndex"] = (
    interaction_df["StockCode"]
    .map(product_to_index)
)

positive_df = interaction_df[
    ["CustomerIndex", "ProductIndex"]
].copy()

positive_df["Purchase"] = 1

positive_pairs = set(
    zip(
        positive_df["CustomerIndex"],
        positive_df["ProductIndex"]
    )
)

np.random.seed(42)

num_customers = len(customers)
num_products = len(products)

negative_samples = []

target_negative_samples = len(positive_df)

while len(negative_samples) < target_negative_samples:
    customer_index = np.random.randint(
        0,
        num_customers
    )

    product_index = np.random.randint(
        0,
        num_products
    )

    if (
        customer_index,
        product_index
    ) not in positive_pairs:
        negative_samples.append(
            [
                customer_index,
                product_index,
                0
            ]
        )

negative_df = pd.DataFrame(
    negative_samples,
    columns=[
        "CustomerIndex",
        "ProductIndex",
        "Purchase"
    ]
)

final_df = pd.concat(
    [
        positive_df,
        negative_df
    ],
    ignore_index=True
)

final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nFinal interaction dataset:")
print(final_df.head())

print(
    "\nTotal interaction records:",
    len(final_df)
)

X_customer = final_df[
    "CustomerIndex"
].values

X_product = final_df[
    "ProductIndex"
].values

y = final_df[
    "Purchase"
].values

(
    X_customer_train,
    X_customer_test,
    X_product_train,
    X_product_test,
    y_train,
    y_test
) = train_test_split(
    X_customer,
    X_product,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

num_customers = len(customers)
num_products = len(products)
embedding_dim = 32

print(
    "\nNumber of customers:",
    num_customers
)

print(
    "Number of products:",
    num_products
)

customer_input = Input(
    shape=(1,),
    name="customer_input"
)

customer_embedding = Embedding(
    input_dim=num_customers,
    output_dim=embedding_dim,
    name="customer_embedding"
)(customer_input)

customer_vector = Flatten()(
    customer_embedding
)

product_input = Input(
    shape=(1,),
    name="product_input"
)

product_embedding = Embedding(
    input_dim=num_products,
    output_dim=embedding_dim,
    name="product_embedding"
)(product_input)

product_vector = Flatten()(
    product_embedding
)

combined = Concatenate()(
    [
        customer_vector,
        product_vector
    ]
)

x = Dense(
    128,
    activation="relu"
)(combined)

x = Dropout(0.3)(x)

x = Dense(
    64,
    activation="relu"
)(x)

x = Dropout(0.2)(x)

x = Dense(
    32,
    activation="relu"
)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(
    inputs=[
        customer_input,
        product_input
    ],
    outputs=output
)

model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nDeep Learning Recommendation Model:")
model.summary()

print("\nTraining Recommendation Model...")

history = model.fit(
    [
        X_customer_train,
        X_product_train
    ],
    y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

test_loss, test_accuracy = model.evaluate(
    [
        X_customer_test,
        X_product_test
    ],
    y_test,
    verbose=0
)

print(
    "\nTest Loss:",
    round(test_loss, 4)
)

print(
    "Test Accuracy:",
    round(test_accuracy * 100, 2),
    "%"
)


def recommend_products(customer_id, top_n=10):

    if customer_id not in customer_to_index:
        print("Customer ID not found.")
        return

    customer_index = customer_to_index[
        customer_id
    ]

    purchased_products = set(
        interaction_df[
            interaction_df["CustomerID"] == customer_id
        ]["StockCode"]
    )

    candidate_products = []

    for product_index in range(num_products):

        product_id = index_to_product[
            product_index
        ]

        if product_id not in purchased_products:
            candidate_products.append(
                product_index
            )

    customer_array = np.array(
        [customer_index] * len(candidate_products)
    )

    product_array = np.array(
        candidate_products
    )

    predictions = model.predict(
        [
            customer_array,
            product_array
        ],
        verbose=0
    ).flatten()

    recommendation_df = pd.DataFrame(
        {
            "StockCode": [
                index_to_product[i]
                for i in candidate_products
            ],
            "Score": predictions
        }
    )

    recommendation_df = (
        recommendation_df
        .sort_values(
            "Score",
            ascending=False
        )
        .head(top_n)
    )

    description_map = (
        df.groupby("StockCode")["Description"]
        .first()
    )

    recommendation_df["Description"] = (
        recommendation_df["StockCode"]
        .map(description_map)
    )

    recommendation_df["Score"] = (
        recommendation_df["Score"] * 100
    ).round(2)

    print(
        "\nRecommended Products for Customer:",
        customer_id
    )

    print(
        recommendation_df[
            [
                "StockCode",
                "Description",
                "Score"
            ]
        ].to_string(index=False)
    )

    return recommendation_df


example_customer = (
    interaction_df["CustomerID"].iloc[0]
)

recommend_products(
    example_customer,
    top_n=10
)

Dataset loaded successfully!
Original dataset shape: (541909, 8)

First 5 records:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  

After cleaning:
Dataset shape: (397884, 8)

Total Sales: 8911407.9

Customer-Product interactions:
   Cus

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ customer_input      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ customer_embedding  │ (None, 1, 32)     │    138,816 │ customer_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_embedding   │ (None, 1, 32)     │    117,280 │ product_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ customer_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ product_embeddin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      8,320 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 274,785 (1.05 MB)

 Trainable params: 274,785 (1.05 MB)

 Non-trainable params: 0 (0.00 B)


Training Recommendation Model...
Epoch 1/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7617 - loss: 0.4868 - val_accuracy: 0.7905 - val_loss: 0.4443
Epoch 2/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7962 - loss: 0.4342 - val_accuracy: 0.7927 - val_loss: 0.4382
Epoch 3/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8015 - loss: 0.4224 - val_accuracy: 0.7939 - val_loss: 0.4361
Epoch 4/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8085 - loss: 0.4114 - val_accuracy: 0.7948 - val_loss: 0.4377
Epoch 5/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8181 - loss: 0.3969 - val_accuracy: 0.7935 - val_loss: 0.4384
Epoch 6/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8304 - loss: 0.3784 - val_accuracy: 0.7926 - val_loss: 0.4429
Epoch 7/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8427 - loss: 0.3576 - val_accuracy: 0.7925 - val_loss: 0.4529
Epoch 8/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.

,StockCode,Score,Description
121,POST,83.199997,POSTAGE
1886,22867,77.190002,HAND WARMER BIRD DESIGN
217,22624,69.300003,IVORY KITCHEN SCALES
1332,23393,68.120003,HOME SWEET HOME CUSHION COVER
777,47590B,66.300003,PINK HAPPY BIRTHDAY BUNTING
253,37449,62.540001,CERAMIC CAKE STAND + HANGING CAKES
1508,84945,60.730000,MULTI COLOUR SILVER T-LIGHT HOLDER
591,20713,60.590000,JUMBO BAG OWLS
2804,23444,59.150002,Next Day Carriage
275,22090,58.139999,PAPER BUNTING RETROSPOT
